<a href="https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

I'm reviewing two findings from FlyRank's March 2026 report, "The State of AI-Driven SEO," the
way I'd want my own Week-5 notebook reviewed: not to grade it, but to ask where the label comes
from and whether the validation carries the claim. The paper is explicit that its ML appendix is
"exploratory" and "secondary to direct aggregate comparisons," which is exactly the right
instinct -- these two questions are in that spirit, aimed at making the exploratory pages even
more trustworthy, not at the paper's headline findings.

**Finding: "What Predicts Health?" (ML Appendix, Random Forest feature importance)**

The paper reports Average Position (43%) and Impressions (32%) as the top predictors of Health
Score, and it already flags the core issue itself: "the target itself is partly constructed from
some of these inputs, so importance is descriptive rather than causal." Health Score's own
formula is Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts).

*My methodology question:* if two of the four scoring inputs are also the top two "predictors,"
this is closer to my Week-5 leakage taxonomy's case 1 (label-derived features) than to genuine
prediction -- a model recovering ~75% of the label's own ingredients isn't finding an external
driver of health, it's rediscovering the formula. Would it be worth reporting a second version
of this chart with Position, Impressions, CTR, and Scroll Depth all excluded, to see whether any
of the remaining features (Content Age, Word Count, AI Sessions, Days Visible) carry any signal
at all once the label's own components are set aside? Right now the chart can't distinguish "the
model found something real" from "the model found the formula," and the paper's own caveat says
as much -- a features-minus-inputs re-run would turn that caveat into a number.

**Finding: "What Predicts Growth?" (ML Appendix, Logistic Regression, 71% holdout accuracy)**

The methodology section states the split as "Logistic Regression (80/20 split)" across "61.8K
content pieces" spanning 57 brands, without saying whether that 80/20 split is row-level or
grouped by brand.

*My methodology question:* pages from the same brand share a CMS, an editorial voice, and
existing search authority -- the exact kind of "hidden character" the leakage skill I'm using
this week warns a random split lets a model memorize. My own Week-5 model showed a measurable
gap (about 10 points of ROC AUC, shown in section 2 below) between a random split and a
brand/client-grouped one on a similar growth-vs-decline task on a related dataset. Was the 71%
holdout accuracy here computed on brands unseen during training, or could some of that number
reflect the model learning per-brand baselines? A second, brand-held-out accuracy figure next to
the current one would let a reader tell the difference -- and if the two numbers are close, that
would actually strengthen the finding.

In [ ]:
import os, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

DATA = "data/raw/content_refresh_anonymized.csv"
if not Path(DATA).exists():
    if not Path("flyrank-ml-internship").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Eng7ouda06/flyrank-ml-internship.git"], check=True)
    os.chdir("flyrank-ml-internship")

df = pd.read_csv(DATA)
df["is_declining_label"] = (df.trend_direction == "down").astype(int)
print(f"{len(df):,} pages | {df.is_declining_label.mean():.1%} declining (base rate) | {df.client_id.nunique()} clients")

# same honest feature set as Week 5: no trend_pct/trend_direction, no last_30d/prev_30d columns
numeric_features = ["impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d", "users_90d",
                    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
                    "days_with_impressions", "days_with_sessions", "content_age_days",
                    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
                    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
                    "word_count", "char_count"]
categorical_features = ["content_type", "main_intent", "competition_level"]

def build_X(frame):
    X_num = frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    cols_with_na = X_num.columns[X_num.isna().any()]
    missing_flags = X_num[cols_with_na].isna().astype(int).add_suffix("_missing")
    X_num = X_num.fillna(0)
    X_cat = pd.get_dummies(frame[categorical_features].fillna("unknown").astype(str), dummy_na=False)
    return pd.concat([X_num.reset_index(drop=True), missing_flags.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)

X = build_X(df)
y = df["is_declining_label"].astype(int)
groups = df["client_id"]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def evaluate(y_true, scores):
    return {"precision@20": precision_at_k(y_true, scores, 20), "precision@50": precision_at_k(y_true, scores, 50),
            "precision@100": precision_at_k(y_true, scores, 100), "roc_auc": roc_auc_score(y_true, scores),
            "avg_precision": average_precision_score(y_true, scores), "base_rate": float(np.mean(y_true))}

def fit_logreg(Xtr, ytr):
    m = Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))])
    m.fit(Xtr, ytr)
    return m

print(f"{X.shape[1]} honest features built")

30,000 pages | 54.2% declining (base rate) | 32 clients
40 honest features built


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Turning the lens on my own Week-5 model. My honest split was already client-grouped, so the
"before" here is what I *didn't* try last week: a plain random 80/20 row split that ignores
`client_id` entirely -- the same shape of split the paper's ML appendix methodology describes.
Same features, same label, same model (Logistic Regression), same random seed. Only the split
changes.

In [ ]:
# BEFORE: naive random row split -- ignores client_id, pages from the same client can appear on both sides
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
m_random = fit_logreg(Xtr_r, ytr_r)
p_random = m_random.predict_proba(Xte_r)[:, 1]
before = evaluate(yte_r, p_random)

# AFTER: honest client-grouped split -- identical to Week 5
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
Xtr_g, Xte_g, ytr_g, yte_g = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]
m_grouped = fit_logreg(Xtr_g, ytr_g)
p_grouped = m_grouped.predict_proba(Xte_g)[:, 1]
after = evaluate(yte_g, p_grouped)

comparison = pd.DataFrame({"random_row_split (BEFORE)": before, "client_grouped_split (AFTER)": after}).T
print(comparison.round(3))

test_clients_random = set(groups.iloc[Xte_r.index])
train_clients_random = set(groups.iloc[Xtr_r.index])
overlap = test_clients_random & train_clients_random
print(f"\nrandom split: {len(overlap)}/{len(test_clients_random)} test-set clients ALSO appear in training")
print(f"grouped split: {groups.iloc[test_idx].nunique()} test-set clients, ALL held out from training")
print(f"\ngap: precision@20 drops {before['precision@20'] - after['precision@20']:.2f}, "
      f"ROC AUC drops {before['roc_auc'] - after['roc_auc']:.3f}, once client memorization is removed")

# --- real failure examples from the honest (grouped) model ---
test_df = df.iloc[test_idx].copy()
test_df["p_grouped"] = p_grouped
test_df["y"] = yte_g.values

cols = ["content_id", "client_id", "p_grouped", "impressions_90d", "avg_position",
        "content_age_days", "days_since_last_update", "trend_direction"]
print("\nconfident but WRONG -- predicted declining, actually up/stable:")
print(test_df[test_df.y == 0].sort_values("p_grouped", ascending=False).head(3)[cols].to_string(index=False))

print("\nconfident but WRONG the other way -- predicted safe, actually declining:")
print(test_df[test_df.y == 1].sort_values("p_grouped", ascending=True).head(3)[cols].to_string(index=False))

                              precision@20  precision@50  precision@100  \
random_row_split (BEFORE)             0.90           0.9           0.83   
client_grouped_split (AFTER)          0.75           0.8           0.74   

                              roc_auc  avg_precision  base_rate  
random_row_split (BEFORE)       0.682          0.701      0.542  
client_grouped_split (AFTER)    0.586          0.589      0.511  

random split: 31/31 test-set clients ALSO appear in training
grouped split: 7 test-set clients, ALL held out from training

gap: precision@20 drops 0.15, ROC AUC drops 0.097, once client memorization is removed

confident but WRONG -- predicted declining, actually up/stable:
          content_id         client_id  p_grouped  impressions_90d  avg_position  content_age_days  days_since_last_update trend_direction
content_374e795aab68 client_f369cb89fc   0.885565              235          31.0               181                      20          stable
content_26d48a980581 

**The gap is the finding.** Under the random split, every single held-out client also has pages
in training (31/31) -- the model can partly recognize "this is client X's kind of page" rather
than learning a generalizable decline signal. That inflates precision@20 from 0.75 to 0.90 and
ROC AUC from 0.586 to 0.682 (+0.097). Neither split is "wrong" to compute, but only the grouped
number is honest about how this model would perform on a brand-new client -- which is the actual
use case. I'm keeping the 0.75 / 0.586 grouped numbers from Week 5 as the real result and
treating the random-split numbers as a measured illustration of how much of a boost client
memorization is worth here, not as a competing headline figure.

The code cell above also pulls the honest model's actual failure cases -- the pages it's most
confidently wrong about in both directions -- so "the model works" isn't just a metric, it's
something I can point at.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same hunt as the Week-3 data contract, run against my final Week-5/6 feature set. First confirm
the banned columns are absent, then deliberately add the worst offender back to prove the test
harness actually catches leakage when it's there -- per the skill: "if it doesn't [jump toward
1.0], your test harness itself is broken."

In [ ]:
banned = ["trend_pct", "trend_direction", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
          "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
present = [c for c in banned if c in X.columns]
print("banned, label-derived columns found in the feature matrix:", present or "none -- clean")

# Deliberately re-add trend_pct (the column the label is thresholded from) and retrain on the SAME grouped split
X_leaky = X.copy()
X_leaky["trend_pct_LEAKY"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)
Xtr_leak, Xte_leak = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]

m_leaky = fit_logreg(Xtr_leak, ytr_g)
p_leaky = m_leaky.predict_proba(Xte_leak)[:, 1]
leaky_metrics = evaluate(yte_g, p_leaky)

audit = pd.DataFrame({"honest (no trend_pct)": after, "WITH trend_pct added back": leaky_metrics}).T
print(audit.round(3))

coefs_leak = pd.Series(m_leaky.named_steps["clf"].coef_[0], index=X_leaky.columns).sort_values(key=abs, ascending=False)
print("\ntop 3 coefficients once the leak is added back (it should tower over everything else):")
print(coefs_leak.head(3).round(3))

# train-without: drop it again, confirm we land back at the honest number
p_check = fit_logreg(Xtr_leak.drop(columns=["trend_pct_LEAKY"]), ytr_g).predict_proba(
    Xte_leak.drop(columns=["trend_pct_LEAKY"]))[:, 1]
print(f"\nAUC after removing it again: {roc_auc_score(yte_g, p_check):.3f} (matches the 0.586 honest baseline)")

banned, label-derived columns found in the feature matrix: none -- clean
                           precision@20  precision@50  precision@100  roc_auc  \
honest (no trend_pct)              0.75           0.8           0.74    0.586   
WITH trend_pct added back          1.00           1.0           1.00    0.999   

                           avg_precision  base_rate  
honest (no trend_pct)              0.589      0.511  
WITH trend_pct added back          0.999      0.511  

top 3 coefficients once the leak is added back (it should tower over everything else):
trend_pct_LEAKY         -49.590
days_with_impressions     0.598
users_90d                -0.407
dtype: float64

AUC after removing it again: 0.586 (matches the 0.586 honest baseline)


banned = ["trend_pct", "trend_direction", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
          "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
present = [c for c in banned if c in X.columns]
print("banned, label-derived columns found in the feature matrix:", present or "none -- clean")

# Deliberately re-add trend_pct (the column the label is thresholded from) and retrain on the SAME grouped split
X_leaky = X.copy()
X_leaky["trend_pct_LEAKY"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)
Xtr_leak, Xte_leak = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]

m_leaky = fit_logreg(Xtr_leak, ytr_g)
p_leaky = m_leaky.predict_proba(Xte_leak)[:, 1]
leaky_metrics = evaluate(yte_g, p_leaky)

audit = pd.DataFrame({"honest (no trend_pct)": after, "WITH trend_pct added back": leaky_metrics}).T
print(audit.round(3))

coefs_leak = pd.Series(m_leaky.named_steps["clf"].coef_[0], index=X_leaky.columns).sort_values(key=abs, ascending=False)
print("\ntop 3 coefficients once the leak is added back (it should tower over everything else):")
print(coefs_leak.head(3).round(3))

# train-without: drop it again, confirm we land back at the honest number
p_check = fit_logreg(Xtr_leak.drop(columns=["trend_pct_LEAKY"]), ytr_g).predict_proba(
    Xte_leak.drop(columns=["trend_pct_LEAKY"]))[:, 1]
print(f"\nAUC after removing it again: {roc_auc_score(yte_g, p_check):.3f} (matches the 0.586 honest baseline)")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest sentence from Week 5 was: *"Logistic Regression wins the metric that matters for a
review queue -- precision@20 = 0.75, precision@50 = 0.80, precision@100 = 0.74, all well above
the 0.51 base rate and clearly ahead of the baseline rule."* That's stated as a flat, general
win. Section 2 above shows exactly why it needs softer edges: the same model's precision@20 was
0.90 under a random split -- 15 points higher on the *same data and label*, just a different
split. If I hadn't checked, I'd have no way to know whether 0.75 was itself still inflated by
some client-level memorization my 7-client test set didn't fully expose.

**Rewrite, in safe language:**

"On a client-grouped holdout (7 clients never seen in training), Logistic Regression showed an
*observed, directional* improvement over the Week-4 rule baseline at precision@20 (0.75 vs.
0.50) and precision@50 (0.80 vs. 0.56). This is *decision-support* evidence for prioritizing a
review queue, not a guaranteed future hit rate -- the same setup produced a measurably higher
number (0.90) under a less honest random split, so the true out-of-sample number for a brand-new
client could plausibly sit lower than 0.75, and should be re-*measured* once real outcomes are
available rather than assumed to hold."

In [ ]:
claims = pd.DataFrame([
    {"version": "Week-5 original (bold)",
     "text": "Logistic Regression wins the metric that matters for a review queue -- precision@20 = 0.75 -- clearly ahead of the baseline rule."},
    {"version": "Week-6 rewrite (safe)",
     "text": "On a client-grouped holdout, Logistic Regression showed an observed, directional improvement over the baseline at precision@20 (0.75 vs 0.50); decision-support evidence, not a guaranteed future hit rate."},
])
import os
os.makedirs("work/outputs", exist_ok=True)
claims.to_csv("work/outputs/w06_claim_rewrite.csv", index=False)
for _, r in claims.iterrows():
    print(f"[{r.version}]\n{r.text}\n")

print(f"reference numbers this rewrite is grounded in: grouped precision@20={after['precision@20']:.2f}, "
      f"random precision@20={before['precision@20']:.2f}, baseline precision@20=0.50 (from Week 5)")

[Week-5 original (bold)]
Logistic Regression wins the metric that matters for a review queue -- precision@20 = 0.75 -- clearly ahead of the baseline rule.

[Week-6 rewrite (safe)]
On a client-grouped holdout, Logistic Regression showed an observed, directional improvement over the baseline at precision@20 (0.75 vs 0.50); decision-support evidence, not a guaranteed future hit rate.

reference numbers this rewrite is grounded in: grouped precision@20=0.75, random precision@20=0.90, baseline precision@20=0.50 (from Week 5)
